In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format  # Display with comma separators and 2 decimal places


In [2]:
df = pd.read_csv("synthetic_customer_churn_data.csv")
df

,account_id,product_id,txn_amt,date_id,txn_flow
0,1675009,14,395.39,6172,debit
1,1006123,28,409.89,6181,debit
2,1936846,74,56.50,6450,debit
3,1542862,104,43.09,6370,debit
4,1648106,97,849.56,6154,debit
...,...,...,...,...,...
9999995,1102857,21,281.49,6617,debit
9999996,1899285,86,799.01,6195,debit
9999997,1807224,65,121.67,6179,debit
9999998,1046974,96,143.07,6055,debit


In [3]:
df.describe()

,account_id,product_id,txn_amt,date_id
count,"10,000,000.00","10,000,000.00","10,000,000.00","10,000,000.00"
mean,"1,499,888.91",79.69,505.05,"6,364.41"
std,"288,708.23",49.65,285.73,210.73
min,"1,000,000.00",10.00,10.00,"6,000.00"
25%,"1,249,755.00",41.00,257.69,"6,182.00"
50%,"1,499,904.00",72.00,504.98,"6,364.00"
75%,"1,749,911.00",103.00,752.53,"6,547.00"
max,"1,999,999.00",210.00,"1,000.00","6,729.00"


In [15]:
df.groupby(by=["txn_flow"]).agg(distinct_accounts= ("account_id","nunique"),txn_cnt=("account_id","count"),txn_amt=("txn_amt","sum"))

,distinct_accounts,txn_cnt,txn_amt
txn_flow,,,
credit,949965,2999930,"1,515,327,350.67"
debit,993258,5000093,"2,525,429,557.69"
services,865093,1999977,"1,009,765,003.20"


In [5]:
max_date_id = df["date_id"].max()
max_date_id

6729

### Let's explore churn

Assumption(bank side) : A customer is called churn if he/she is inactive for atleast 30 days

Initially we create segment's based on 30 days durations<br>
segment1 : Initial 30 days<br>
segment2 : 31-60 days<br>
segment3 : 61-90 days<br>
segment4 : 91-120 days (optional)

based on transaction pattern obtained from segment2 and segment3 we try to predict if account will be active in segment1 

In [240]:
max_date_id = max_date_id-30
duration = 30
segment1 = max_date_id - duration
segment2 = segment1 - duration
segment3 = segment2 - duration

print(f" days : {max_date_id= },{segment1=},{segment2=},{segment3=}")

 days : max_date_id= 6699,segment1=6669,segment2=6639,segment3=6609


In [241]:
df.head()

,account_id,product_id,txn_amt,date_id,txn_flow
0,1675009,14,395.39,6172,debit
1,1006123,28,409.89,6181,debit
2,1936846,74,56.50,6450,debit
3,1542862,104,43.09,6370,debit
4,1648106,97,849.56,6154,debit


In [245]:
df_recent = df.loc[(df.date_id >= segment3) & (df.date_id <= max_date_id)]
df_recent.head()

,account_id,product_id,txn_amt,date_id,txn_flow
14,1219814,91,34.71,6651,debit
20,1263226,54,927.72,6667,debit
21,1357131,55,997.18,6671,debit
39,1555229,39,646.28,6653,debit
46,1235855,42,953.42,6683,debit


In [246]:
df_recent["segment"] = pd.cut(x=df_recent.date_id,bins=[segment3,segment2,segment1,max_date_id],labels=[3,2,1],include_lowest=False
                              )
df_recent

/tmp/ipykernel_6381/2290748173.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_recent["segment"] = pd.cut(x=df_recent.date_id,bins=[segment3,segment2,segment1,max_date_id],labels=[3,2,1],include_lowest=False


,account_id,product_id,txn_amt,date_id,txn_flow,segment
14,1219814,91,34.71,6651,debit,2
20,1263226,54,927.72,6667,debit,2
21,1357131,55,997.18,6671,debit,1
39,1555229,39,646.28,6653,debit,2
46,1235855,42,953.42,6683,debit,1
...,...,...,...,...,...,...
9999952,1719112,70,393.44,6633,debit,3
9999973,1586878,103,422.78,6668,debit,2
9999977,1160790,92,322.27,6666,debit,2
9999982,1321796,24,614.14,6625,debit,3


In [247]:
df_recent.groupby("segment").agg(min_value = ("date_id","min"),max_value=("date_id","max"),accounts=("account_id","nunique"))

,min_value,max_value,accounts
segment,,,
3,6610,6639,336793
2,6640,6669,336291
1,6670,6699,336032


In [248]:
facts = df_recent.groupby(by=["account_id","segment","txn_flow"]).agg(txn_cnt=("account_id","count"),txn_amt=("txn_amt","sum")).reset_index()
facts.head(10)

,account_id,segment,txn_flow,txn_cnt,txn_amt
0,1000002,3,credit,0,0.00
1,1000002,3,debit,1,492.24
2,1000002,3,services,0,0.00
3,1000002,2,credit,0,0.00
4,1000002,2,debit,0,0.00
5,1000002,2,services,0,0.00
6,1000002,1,credit,0,0.00
7,1000002,1,debit,0,0.00
8,1000002,1,services,0,0.00
9,1000003,3,credit,0,0.00


In [249]:
mapping_txn_flow = {"not_present":0,"credit":1,"debit":2,"services":3,"credit_debit":4,"credit_services":5,"debit_services":6,"credit_debit_service":7}
master_txn_flow = pd.DataFrame(mapping_txn_flow.items(),columns=["txn_flow_name","id"])
master_txn_flow


,txn_flow_name,id
0,not_present,0
1,credit,1
2,debit,2
3,services,3
4,credit_debit,4
5,credit_services,5
6,debit_services,6
7,credit_debit_service,7


In [250]:
facts_p = facts.pivot(index=["account_id","segment"],columns=["txn_flow"],values=["txn_cnt"])
facts_p

txn_cnt               
txn_flow            credit debit services
account_id segment                       
1000002    3             0     1        0
           2             0     0        0
           1             0     0        0
1000003    3             0     1        0
           2             0     0        0
...                    ...   ...      ...
1999997    2             0     0        0
           1             0     0        0
1999998    3             0     0        0
           2             0     0        1
           1             0     0        0

[2136762 rows x 3 columns]

In [251]:
facts_p.columns = facts_p.columns.droplevel(0)  # Drop the first level ('txn_cnt')
facts_p.columns.name=None
facts_p = facts_p.reset_index()
facts_p


,account_id,segment,credit,debit,services
0,1000002,3,0,1,0
1,1000002,2,0,0,0
2,1000002,1,0,0,0
3,1000003,3,0,1,0
4,1000003,2,0,0,0
...,...,...,...,...,...
2136757,1999997,2,0,0,0
2136758,1999997,1,0,0,0
2136759,1999998,3,0,0,0
2136760,1999998,2,0,0,1


In [279]:
p = facts_p.groupby(by=["segment","txn_flow"]).agg(account_cnt=("account_id","nunique")).reset_index()
p["txn_flow"] = p["txn_flow"].astype("int")


pp=p.merge(master_txn_flow, left_on="txn_flow",right_on="id").drop("id",axis=1)
pp.sort_values(by=['segment', 'txn_flow'],ascending=[False,True])


,segment,txn_flow,account_cnt,txn_flow_name
2,1,0,376222,not_present
5,1,1,86707,credit
8,1,2,151061,debit
11,1,3,56586,services
14,1,4,19628,credit_debit
17,1,5,7396,credit_services
20,1,6,12957,debit_services
23,1,7,1697,credit_debit_service
1,2,0,375963,not_present
4,2,1,87110,credit


In [255]:
final_facts = facts_p.pivot(index="account_id",columns="segment",values="txn_flow")
final_facts.columns.name=None
final_facts.reset_index(inplace=True)
final_facts


,account_id,3,2,1
0,1000002,2,0,0
1,1000003,2,0,6
2,1000004,4,0,0
3,1000005,0,0,2
4,1000006,0,0,1
...,...,...,...,...
712249,1999994,2,0,0
712250,1999995,4,1,0
712251,1999996,0,0,1
712252,1999997,2,0,0


In [256]:
final_facts.columns = ["account_id","3","2","1"]
final_facts = final_facts.rename(columns={"1":"current","2":"last","3":"second_last"})

final_facts["current"] = final_facts["current"].astype("int")

final_facts["last"] = final_facts["last"].astype("str")
final_facts["second_last"] = final_facts["second_last"].astype("str")


final_facts["seq"] = final_facts["second_last"]+final_facts["last"]
final_facts["seq"] = final_facts["seq"].astype("str")
final_facts["is_churn"] = (final_facts["current"]==0).astype("int")
final_facts


,account_id,second_last,last,current,seq,is_churn
0,1000002,2,0,0,20,1
1,1000003,2,0,6,20,0
2,1000004,4,0,0,40,1
3,1000005,0,0,2,00,0
4,1000006,0,0,1,00,0
...,...,...,...,...,...,...
712249,1999994,2,0,0,20,1
712250,1999995,4,1,0,41,1
712251,1999996,0,0,1,00,0
712252,1999997,2,0,0,20,1


#### Facts Descriptions
- seq  : Concatenation of second_last and last facts
- is_churn : 1 if current == 0 else 1

## Result for churn accounts

In [286]:
master_result = final_facts.query("current==0").groupby(["second_last","last","seq"]).agg(cnt=("account_id","count")).reset_index()
master_result.sort_values("cnt",ascending=False,inplace=True)
master_result.reset_index(drop=True, inplace=True)

master_result["cummulative_cnt"] = master_result["cnt"].cumsum()
total_churn_accounts = master_result["cummulative_cnt"].iloc[-1]
print(f"total churn accounts : {total_churn_accounts}")
master_result["relative_churn_per"] = (master_result["cnt"] / total_churn_accounts)*100
master_result["cummulative_churn_per"] = (master_result["cummulative_cnt"] / total_churn_accounts)*100
master_result.head(20)

total churn accounts : 376222


,second_last,last,seq,cnt,cummulative_cnt,relative_churn_per,cummulative_churn_per
0,2,0,20,66588,66588,17.70,17.70
1,0,2,02,66250,132838,17.61,35.31
2,1,0,10,38536,171374,10.24,45.55
3,0,1,01,38327,209701,10.19,55.74
4,3,0,30,25313,235014,6.73,62.47
5,0,3,03,24975,259989,6.64,69.11
6,2,2,22,15165,275154,4.03,73.14
7,2,1,21,8797,283951,2.34,75.47
8,0,4,04,8729,292680,2.32,77.79
9,1,2,12,8704,301384,2.31,80.11


In [287]:

master_result = final_facts.groupby(by=["seq","is_churn"]).agg(cnt=("account_id","count")).reset_index()
master_result.reset_index(drop=True, inplace=True)
master_result = master_result.pivot(index="seq",columns="is_churn",values="cnt").fillna(0).reset_index()
master_result.columns.name=None

master_result.columns = ["seq","not_churn","churn"]
master_result.sort_values("churn",ascending=False,inplace=True)

master_result = master_result.astype({"churn":"int","not_churn":"int"})
master_result["total_accounts"] = master_result["churn"] + master_result["not_churn"]

master_result["churn_percentage"] = (master_result["churn"]/master_result["total_accounts"])*100
# master_result.sort_values("churn_percentage",ascending=True,inplace=True)

master_result["cummulative_churn_acc"] = master_result["churn"].cumsum()

total_churn_accounts = master_result["cummulative_churn_acc"].iloc[-1]

# master_result["relative_churn_per"] = (master_result["cnt"] / total_churn_accounts)*100
# master_result["cummulative_churn_per"] = (master_result["cummulative_cnt"] / total_churn_accounts)*100
master_result.head(20)

,seq,not_churn,churn,total_accounts,churn_percentage,cummulative_churn_acc
16,20,33585,66588,100173,66.47,66588
2,02,33615,66250,99865,66.34,132838
8,10,19348,38536,57884,66.57,171374
1,01,19464,38327,57791,66.32,209701
24,30,12582,25313,37895,66.80,235014
3,03,12745,24975,37720,66.21,259989
18,22,7435,15165,22600,67.10,275154
17,21,4309,8797,13106,67.12,283951
4,04,4442,8729,13171,66.27,292680
10,12,4371,8704,13075,66.57,301384


Only consider those sequence where churn account count >= 500.<br>
If we want to have churn prediction for 90percentage of churn accounts then we can take 23 different sequences(combination) where cummulative churn percentage value is 90

In [308]:
final_master_result = master_result.query("churn>=500").sort_values("churn_percentage",ascending=False)
final_master_result["cummulative_churn_acc"] = final_master_result["churn"].cumsum()
final_master_result["cummulative_churn_per"] = (final_master_result["cummulative_churn_acc"] / total_churn_accounts)*100
final_master_result.reset_index(drop=True,inplace=True)
final_master_result["second_last"] = (final_master_result["seq"].str[0]).astype("int")
final_master_result["last"] = (final_master_result["seq"].str[1]).astype("int")

final_master_result = final_master_result.merge(master_txn_flow,left_on="second_last",right_on="id",how="left").drop(["second_last","id"],axis=1).rename(columns={"txn_flow_name":"second_last"})
final_master_result = final_master_result.merge(master_txn_flow,left_on="last",right_on="id",how="left").drop(["last","id"],axis=1).rename(columns={"txn_flow_name":"last"})

# final_master_result = final_master_result.rename(columns={"txn_flow_name":"second_last"})
final_master_result = final_master_result[final_master_result.columns[-2:].to_list() + final_master_result.columns[:-2].to_list()]

final_master_result


,second_last,last,seq,not_churn,churn,total_accounts,churn_percentage,cummulative_churn_acc,cummulative_churn_per
0,not_present,credit_debit_service,07,362,756,1118,67.62,756,0.20
1,not_present,credit_services,05,1584,3293,4877,67.52,4049,1.08
2,debit,debit_services,26,636,1300,1936,67.15,5349,1.42
3,debit,credit,21,4309,8797,13106,67.12,14146,3.76
4,debit,debit,22,7435,15165,22600,67.10,29311,7.79
5,debit_services,credit,61,369,749,1118,66.99,30060,7.99
6,services,debit,32,2819,5719,8538,66.98,35779,9.51
7,services,not_present,30,12582,25313,37895,66.80,61092,16.24
8,credit,credit,11,2502,5016,7518,66.72,66108,17.57
9,services,services,33,1069,2143,3212,66.72,68251,18.14


In [309]:
final_master_result = final_master_result.query("cummulative_churn_per < 91")
final_master_result

,second_last,last,seq,not_churn,churn,total_accounts,churn_percentage,cummulative_churn_acc,cummulative_churn_per
0,not_present,credit_debit_service,07,362,756,1118,67.62,756,0.20
1,not_present,credit_services,05,1584,3293,4877,67.52,4049,1.08
2,debit,debit_services,26,636,1300,1936,67.15,5349,1.42
3,debit,credit,21,4309,8797,13106,67.12,14146,3.76
4,debit,debit,22,7435,15165,22600,67.10,29311,7.79
5,debit_services,credit,61,369,749,1118,66.99,30060,7.99
6,services,debit,32,2819,5719,8538,66.98,35779,9.51
7,services,not_present,30,12582,25313,37895,66.80,61092,16.24
8,credit,credit,11,2502,5016,7518,66.72,66108,17.57
9,services,services,33,1069,2143,3212,66.72,68251,18.14


In [310]:
final_master_result.to_parquet("result/churn_eda_master.parquet",index=False)

## Algorithm for Identifying Churn Patterns

### Step 1: Preprocessing and Segmentation
1. Discard the latest 30 days of data.
2. Create a `segment` fact based on `date_id` to identify different nature of accounts during those time periods:
   - `1` if `date_id` is between `6670` and `6699` (first 30 days)
   - `2` if `date_id` is between `6640` and `6669` (30 to 60 days)
   - `3` if `date_id` is between `6610` and `6639` (60 to 90 days)

### Step 2: Aggregate Transaction Data
1. Group by `account_id`, `segment`, and `txn_flow`.
2. Compute:
   - `txn_cnt`: Count of `account_id`
   - `txn_amt`: Sum of transaction amounts

### Step 3: Reclassify Transaction Flow
Modify `txn_flow` to new categories:

| txn_flow Code | Condition |
|--------------|------------------------------------------------|
| 0 (not_present) | `credit == 0` and `debit == 0` and `services == 0` |
| 1 (credit) | `credit >= 1` and `debit == 0` and `services == 0` |
| 2 (debit) | `credit == 0` and `debit >= 1` and `services == 0` |
| 3 (services) | `credit == 0` and `debit == 0` and `services >= 1` |
| 4 (credit_debit) | `credit >= 1` and `debit >= 1` and `services == 0` |
| 5 (credit_services) | `credit >= 1` and `debit == 0` and `services >= 1` |
| 6 (debit_services) | `credit == 0` and `debit >= 1` and `services >= 1` |
| 7 (credit_debit_services) | `credit >= 1` and `debit >= 1` and `services >= 1` |

### Step 4: Create Churn Sequence Facts
1. Rename segments:
   - `3 → second_last`
   - `2 → last`
   - `1 → current`
2. Define churn-related facts:
   - `seq` = Concatenation of `second_last + last`
   - `is_churn` = `1` if `current == 0`, else `0`

### Step 5: Compute Churn Statistics
1. Group by `seq` and `is_churn` to count:
   - `churn_accounts`
   - `non_churn_accounts`
2. Compute additional churn metrics:
   - `total_accounts = churn + non_churn`
   - `churn_percentage = churn / total_accounts`
   - `cumulative_churn_acc = cumulative sum of churn`
   - `cumulative_churn_per = cumulative_churn_acc / total_churn_accounts`

### Step 6: Identify High-Risk Churn Sequences
1. Apply filter: `cumulative_churn_per < 91 and churn >= 500`
2. Extract **23 sequences** that account for **90% churn**.
3. Store the results as `master_churn` for future predictions.

---

## Algorithm for Churn Prediction

### Step 1: Preprocessing and Segmentation
1. Create a `segment` fact based on `date_id`:
   - `1` if `date_id` is between `6700` and `6729` (first 30 days)
   - `2` if `date_id` is between `6670` and `6699` (30 to 60 days)
   - `3` if `date_id` is between `6640` and `6669` (60 to 90 days)

### Step 2: Aggregate Transaction Data
1. Group by `account_id`, `segment`, and `txn_flow`.
2. Compute:
   - `txn_cnt`: Count of `account_id`
   - `txn_amt`: Sum of transaction amounts

### Step 3: Reclassify Transaction Flow
(Use the same transaction flow categorization as in Step 3 of the churn pattern algorithm.)

### Step 4: Create Churn Sequence Facts
1. Rename segments:
   - `3 → second_last`
   - `2 → last`
   - `1 → current`
2. Define churn-related facts:
   - `seq` = Concatenation of `second_last + last`
   - `is_churn` = `1` if `current == 0`, else `0`

### Step 5: Predict Churn
1. Read data from `master_churn`.
2. If `seq` from current data is in `master_churn`, then `predicted_churn = 1`, else `0`.
3. Compare `predicted_churn` with `is_churn`.
4. Generate following metrics to evaluate our prediction:
   - Classification report
   - Confusion matrix
